In [1]:
# ============================================================
# CHECKPOINT 0: Load 1 file nhỏ từ HuggingFace, verify schema
# Mục tiêu: biết đúng tên cột trước khi làm gì tiếp theo
# ============================================================

!pip install huggingface_hub pyarrow -q

from huggingface_hub import snapshot_download, list_repo_files
import pyarrow.parquet as pq
import pandas as pd
import os

REPO_ID   = "datdong2004/amazonNew-cleaned"
CATEGORY  = "electronics"
HF_TOKEN  = None  # nếu repo private thì điền token vào đây

# ── Step 1: Xem repo có những file/folder gì ─────────────────
print("=== Cấu trúc repo ===")
try:
    files = list(list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN))
    for f in files[:30]:
        print(f)
    if len(files) > 30:
        print(f"... và {len(files)-30} files nữa")
except Exception as e:
    print(f"Lỗi: {e}")

=== Cấu trúc repo ===
.gitattributes
checkpoints.json
main_category=alexa_skills/._SUCCESS.crc
main_category=alexa_skills/_SUCCESS
main_category=alexa_skills/overall=1/.part-00000-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00001-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00002-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00003-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00004-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00005-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00006-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_category=alexa_skills/overall=1/.part-00007-b2365165-ba0e-496a-92fd-ac1a60cbfe57.c000.zstd.parquet.crc
main_

In [2]:
# ── Step 2: Download chỉ Electronics category ────────────────
# ignore_patterns để không kéo về categories khác — tiết kiệm thời gian và storage

LOCAL_DIR = f"/kaggle/working/data/{CATEGORY}"
os.makedirs(LOCAL_DIR, exist_ok=True)

snapshot_download(
    repo_id    = REPO_ID,
    repo_type  = "dataset",
    local_dir  = LOCAL_DIR,
    token      = HF_TOKEN,
    # Chỉ lấy Electronics, bỏ qua tất cả categories khác
    allow_patterns  = [f"*Electronics*", f"*electronics*"],
    ignore_patterns = ["*.json", "*.md", "*.txt"],
)

# Kiểm tra đã download được gì
downloaded = []
for root, dirs, fnames in os.walk(LOCAL_DIR):
    for f in fnames:
        if f.endswith(".parquet"):
            full = os.path.join(root, f)
            size_mb = os.path.getsize(full) / 1024**2
            downloaded.append((full, size_mb))

print(f"\nĐã download: {len(downloaded)} files")
print(f"Tổng dung lượng: {sum(s for _, s in downloaded):.1f} MB")
for path, mb in downloaded[:5]:
    rel = os.path.relpath(path, LOCAL_DIR)
    print(f"  {rel:60s}  {mb:.1f} MB")

Fetching ... files: 0it [00:00, ?it/s]


Đã download: 80 files
Tổng dung lượng: 5684.3 MB
  main_category=electronics/overall=3/batch002-part-00005-f3eb2ebc-56b9-42d5-a4e6-451db19dd9ea.c000.zstd.parquet  12.1 MB
  main_category=electronics/overall=3/batch002-part-00007-f3eb2ebc-56b9-42d5-a4e6-451db19dd9ea.c000.zstd.parquet  12.1 MB
  main_category=electronics/overall=3/batch001-part-00000-d11fee19-a34b-47ac-9c51-68ee3cbba983.c000.zstd.parquet  47.5 MB
  main_category=electronics/overall=3/batch002-part-00001-f3eb2ebc-56b9-42d5-a4e6-451db19dd9ea.c000.zstd.parquet  12.1 MB
  main_category=electronics/overall=3/batch002-part-00002-f3eb2ebc-56b9-42d5-a4e6-451db19dd9ea.c000.zstd.parquet  12.2 MB


In [3]:
import pyarrow.dataset as ds
import pyarrow as pa
import pandas as pd

# Dùng ds.field trực tiếp, không qua pq.read_table
# explicit schema cho partition columns để fix type conflict

partition_schema = pa.schema([
    pa.field("overall", pa.int32()),
])

dataset = ds.dataset(
    LOCAL_DIR,
    format="parquet",
    partitioning=ds.partitioning(
        partition_schema,
        flavor="hive"  # tự parse overall=1/, overall=2/, ...
    ),
    # Fix main_category conflict: ép tất cả về string
    schema=None,
)

# Thử đọc 5 rows để confirm overall xuất hiện
df_test = dataset.head(5).to_pandas()
print("Columns:", df_test.columns.tolist())
print("\nOverall:", df_test["overall"].tolist() if "overall" in df_test.columns else "MISSING")
print("\nSample:")
print(df_test[["asin", "reviewerID", "overall", "review_date"]].to_string())

Columns: ['asin', 'reviewerID', 'reviewText', 'summary', 'title', 'price', 'brand', 'category', 'text', 'review_date', 'price_clean', 'flag_malformed_asin', 'flag_malformed_reviewer', 'flag_non_ascii_review', 'flag_malformed_overall', 'flag_malformed_date', 'main_category', 'flag_short_review', 'flag_no_prod_desc', 'flag_missing_price', 'flag_extreme_rating', 'overall']

Overall: [1, 1, 1, 1, 1]

Sample:
         asin      reviewerID  overall review_date
0  B00IADP2IO  A232QU7MCAQD51        1  2016-05-29
1  B00006JPFU  A2G2SAM49N7KF8        1  2008-11-30
2  B01E4KCFRC   ANP11UOHEQ4OM        1  2016-05-31
3  B00QJ3MT56  A13SWX424MX0WT        1  2015-05-11
4  B015JV4QHC   A2N910W3QAURT        1  2018-07-01


# **CHECKPOINT 1 — Load Electronics & Temporal Split**

In [4]:
import re

COL_USER   = "reviewerID"
COL_ITEM   = "asin"
COL_RATING = "overall"
COL_DATE   = "review_date"
NEEDED     = [COL_USER, COL_ITEM, COL_DATE]  # overall từ partition

# Load tất cả files, parse overall từ folder name
dfs = []
for path, size_mb in downloaded:
    match = re.search(r'overall=(\d)', path)
    overall_val = int(match.group(1)) if match else None
    pf = pq.ParquetFile(path)
    df_chunk = pf.read(columns=NEEDED).to_pandas()
    df_chunk[COL_RATING] = overall_val
    dfs.append(df_chunk)

df = pd.concat(dfs, ignore_index=True)

# Đảm bảo kiểu dữ liệu đúng
df[COL_DATE]   = pd.to_datetime(df[COL_DATE])
df[COL_RATING] = df[COL_RATING].astype(int)

print(f"Total reviews : {len(df):,}")
print(f"RAM           : ~{df.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"Null counts   :\n{df.isnull().sum()}")
print(f"\nRating distribution:")
print(df[COL_RATING].value_counts().sort_index())
print(f"\nDate range: {df[COL_DATE].min()} → {df[COL_DATE].max()}")

Total reviews : 12,959,002
RAM           : ~1.78 GB
Null counts   :
reviewerID     0
asin           0
review_date    0
overall        0
dtype: int64

Rating distribution:
overall
1    1202047
2     689897
3     993363
4    2116404
5    7957291
Name: count, dtype: int64

Date range: 1999-04-13 00:00:00 → 2018-10-05 00:00:00


In [5]:
# ============================================================
# K-CORE FILTERING — chạy trước CP1, thay thế df gốc
# ============================================================
K = 5

print(f"Trước filtering: {len(df):,} reviews, "
      f"{df[COL_USER].nunique():,} users, "
      f"{df[COL_ITEM].nunique():,} items")

df_core = df.copy()
iteration = 0

while True:
    iteration += 1
    
    # Đếm reviews per user và per item trong current df
    user_counts = df_core[COL_USER].value_counts()
    item_counts = df_core[COL_ITEM].value_counts()
    
    # Giữ lại chỉ users và items đủ K reviews
    valid_users = user_counts[user_counts >= K].index
    valid_items = item_counts[item_counts >= K].index
    
    df_filtered = df_core[
        df_core[COL_USER].isin(valid_users) &
        df_core[COL_ITEM].isin(valid_items)
    ]
    
    n_removed = len(df_core) - len(df_filtered)
    print(f"Iter {iteration}: removed {n_removed:,} reviews → {len(df_filtered):,} remaining")
    
    # Dừng khi không còn gì bị remove
    if n_removed == 0:
        break
    
    df_core = df_filtered

print(f"\nSau {K}-core filtering:")
print(f"  Reviews : {len(df_core):,}  ({len(df_core)/len(df):.1%} của original)")
print(f"  Users   : {df_core[COL_USER].nunique():,}")
print(f"  Items   : {df_core[COL_ITEM].nunique():,}")
print(f"\nReviews per user sau filtering:")
print(df_core.groupby(COL_USER).size().describe(percentiles=[.5,.75,.9,.95]))
print(f"\nReviews per item sau filtering:")
print(df_core.groupby(COL_ITEM).size().describe(percentiles=[.5,.75,.9,.95]))

Trước filtering: 12,959,002 reviews, 4,645,572 users, 245,740 items
Iter 1: removed 7,055,660 reviews → 5,903,342 remaining
Iter 2: removed 229,629 reviews → 5,673,713 remaining
Iter 3: removed 151,136 reviews → 5,522,577 remaining
Iter 4: removed 13,439 reviews → 5,509,138 remaining
Iter 5: removed 9,229 reviews → 5,499,909 remaining
Iter 6: removed 860 reviews → 5,499,049 remaining
Iter 7: removed 632 reviews → 5,498,417 remaining
Iter 8: removed 44 reviews → 5,498,373 remaining
Iter 9: removed 40 reviews → 5,498,333 remaining
Iter 10: removed 0 reviews → 5,498,333 remaining

Sau 5-core filtering:
  Reviews : 5,498,333  (42.4% của original)
  Users   : 609,641
  Items   : 145,785

Reviews per user sau filtering:
count    609641.000000
mean          9.018969
std           7.889936
min           5.000000
50%           7.000000
75%          10.000000
90%          15.000000
95%          20.000000
max         572.000000
dtype: float64

Reviews per item sau filtering:
count    145785.00000

In [6]:
# Redo CP1 với df_core
df_core = df_core.sort_values(COL_DATE).reset_index(drop=True)

split_idx  = int(len(df_core) * 0.8)
split_date = df_core.iloc[split_idx][COL_DATE]

train = df_core.iloc[:split_idx].copy()
test  = df_core.iloc[split_idx:].copy()

train_users = set(train[COL_USER].unique())
train_items = set(train[COL_ITEM].unique())

test = test.copy()
test["user_known"] = test[COL_USER].isin(train_users)
test["item_known"] = test[COL_ITEM].isin(train_items)

mask_warm    = test["user_known"] & test["item_known"]
mask_cold_u  = ~test["user_known"] & test["item_known"]
mask_cold_i  = test["user_known"] & ~test["item_known"]
mask_cold_ui = ~test["user_known"] & ~test["item_known"]

n_users  = df_core[COL_USER].nunique()
n_items  = df_core[COL_ITEM].nunique()
sparsity = 1 - len(df_core) / (n_users * n_items)

print(f"Split date : {split_date}")
print(f"Train      : {len(train):,}")
print(f"Test       : {len(test):,}")
print(f"Sparsity   : {sparsity:.6%}")
print(f"\nWarm (both known)         : {mask_warm.sum():>8,}  ({mask_warm.mean():.1%})")
print(f"Cold user, warm item      : {mask_cold_u.sum():>8,}  ({mask_cold_u.mean():.1%})")
print(f"Warm user, cold item      : {mask_cold_i.sum():>8,}  ({mask_cold_i.mean():.1%})")
print(f"Fully cold (both unknown) : {mask_cold_ui.sum():>8,}  ({mask_cold_ui.mean():.1%})")

Split date : 2016-12-09 00:00:00
Train      : 4,398,666
Test       : 1,099,667
Sparsity   : 99.993814%

Warm (both known)         :  905,868  (82.4%)
Cold user, warm item      :  156,614  (14.2%)
Warm user, cold item      :   32,454  (3.0%)
Fully cold (both unknown) :    4,731  (0.4%)


"After 5-core filtering, 82.4% of test pairs are warm. The remaining 14.2% cold-user pairs are users whose review history falls entirely within the test window — an unavoidable artifact of temporal splitting on a dataset with median user activity of 7 reviews. Cold-start cases are evaluated separately and addressed in Stage X."

# **CP2 — Compute Means**

In [7]:
global_mean = train[COL_RATING].mean()
global_std  = train[COL_RATING].std()
print(f"Global mean : {global_mean:.4f}")
print(f"Global std  : {global_std:.4f}")

user_stats = train.groupby(COL_USER)[COL_RATING].agg(["mean","count"])
user_stats.columns = ["user_mean_raw", "user_n"]

item_stats = train.groupby(COL_ITEM)[COL_RATING].agg(["mean","count"])
item_stats.columns = ["item_mean_raw", "item_n"]

print(f"\nUser review count — median: {user_stats['user_n'].median():.0f}, mean: {user_stats['user_n'].mean():.1f}")
print(f"Item review count — median: {item_stats['item_n'].median():.0f}, mean: {item_stats['item_n'].mean():.1f}")
print(f"\nUsers với 1 review: {(user_stats['user_n']==1).sum():,} ({(user_stats['user_n']==1).mean():.1%})")
print(f"Items với 1 review: {(item_stats['item_n']==1).sum():,} ({(item_stats['item_n']==1).mean():.1%})")

Global mean : 4.2318
Global std  : 1.2156

User review count — median: 6, mean: 7.5
Item review count — median: 10, mean: 30.7

Users với 1 review: 16,877 (2.9%)
Items với 1 review: 2,770 (1.9%)


In [8]:
# Bayesian shrinkage
m_user = float(user_stats["user_n"].median())
m_item = float(item_stats["item_n"].median())

user_stats["user_mean"] = (
    user_stats["user_n"] * user_stats["user_mean_raw"] + m_user * global_mean
) / (user_stats["user_n"] + m_user)

item_stats["item_mean"] = (
    item_stats["item_n"] * item_stats["item_mean_raw"] + m_item * global_mean
) / (item_stats["item_n"] + m_item)

print(f"m_user={m_user:.1f}, m_item={m_item:.1f}")
print(f"\nUser mean (Bayesian): {user_stats['user_mean'].describe()}")
print(f"\nItem mean (Bayesian): {item_stats['item_mean'].describe()}")

m_user=6.0, m_item=10.0

User mean (Bayesian): count    586014.000000
mean          4.222381
std           0.345819
min           1.775634
25%           4.035533
50%           4.298858
75%           4.490078
max           4.965858
Name: user_mean, dtype: float64

Item mean (Bayesian): count    143281.000000
mean          4.191231
std           0.301251
min           2.033161
25%           4.021207
50%           4.253243
75%           4.409085
max           4.976722
Name: item_mean, dtype: float64


In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

user_mean_dict = user_stats["user_mean"].to_dict()
item_mean_dict = item_stats["item_mean"].to_dict()

def predict_vectorized(df_subset):
    u = df_subset[COL_USER].map(user_mean_dict).fillna(global_mean)
    i = df_subset[COL_ITEM].map(item_mean_dict).fillna(global_mean)
    return ((u + i) / 2.0).clip(1.0, 5.0)

test["pred"]   = predict_vectorized(test)
test["actual"] = test[COL_RATING]

def evaluate(subset, label):
    if len(subset) == 0:
        print(f"{label:35s}: no samples")
        return
    rmse = mean_squared_error(subset["actual"], subset["pred"]) ** 0.5  # fix ở đây
    mae  = mean_absolute_error(subset["actual"], subset["pred"])
    print(f"{label:35s}  n={len(subset):>8,}  RMSE={rmse:.4f}  MAE={mae:.4f}")

print("="*75)
print("STAGE 0 — ELECTRONICS BASELINE")
print("="*75)
evaluate(test,             "ALL TEST")
evaluate(test[mask_warm],  "Warm (both known)")
evaluate(test[mask_cold_u],"Cold user, warm item")
evaluate(test[mask_cold_i],"Warm user, cold item")
evaluate(test[mask_cold_ui],"Fully cold")

# Error distribution
errors = (test["actual"] - test["pred"]).abs()
print(f"\nError distribution:")
print(errors.describe(percentiles=[.5,.75,.9,.95,.99]))

big_err = (errors > 2.0).sum()
print(f"\nSai > 2 sao: {big_err:,} ({big_err/len(test):.2%})")

# Per-star breakdown (warm only)
print(f"\nRMSE per star (warm):")
warm = test[mask_warm]
for star in sorted(warm["actual"].unique()):
    s = warm[warm["actual"]==star]
    rmse = mean_squared_error(s["actual"], s["pred"]) ** 0.5
    mae  = mean_absolute_error(s["actual"], s["pred"])
    print(f"  ★{int(star)}  n={len(s):>7,}  RMSE={rmse:.4f}  MAE={mae:.4f}")

STAGE 0 — ELECTRONICS BASELINE
ALL TEST                             n=1,099,667  RMSE=1.1976  MAE=0.9167
Warm (both known)                    n= 905,868  RMSE=1.2134  MAE=0.9251
Cold user, warm item                 n= 156,614  RMSE=1.0987  MAE=0.8617
Warm user, cold item                 n=  32,454  RMSE=1.2140  MAE=0.9470
Fully cold                           n=   4,731  RMSE=1.1655  MAE=0.9310

Error distribution:
count    1.099667e+06
mean     9.167483e-01
std      7.705658e-01
min      4.457062e-06
50%      6.750062e-01
75%      9.392980e-01
90%      2.199881e+00
95%      3.051889e+00
99%      3.385383e+00
max      3.797062e+00
dtype: float64

Sai > 2 sao: 126,350 (11.49%)

RMSE per star (warm):
  ★1  n= 75,328  RMSE=3.1094  MAE=3.0982
  ★2  n= 45,817  RMSE=2.1414  MAE=2.1279
  ★3  n= 66,502  RMSE=1.1942  MAE=1.1728
  ★4  n=132,803  RMSE=0.3094  MAE=0.2675
  ★5  n=585,418  RMSE=0.7012  MAE=0.6724


Vấn đề: baseline predict gần như mọi thứ quanh 4.0–4.3 (vì global mean và hầu hết user/item means đều cluster ở đó). ★4 và ★5 chiếm ~79% warm test nên kéo overall RMSE xuống trông có vẻ ổn. Nhưng ★1 và ★2 bị predict sai hoàn toàn — đây là class imbalance problem.
Cold users (1.0986 RMSE) trông tốt hơn warm vì phần lớn cold users là người mới, có xu hướng rate cao (họ mua và hài lòng mới review), nên predict global_mean 4.23 cho họ lại ít sai hơn.

Kết luận cho Stage 0
Kết quả nằm trong expected range, không có bug. Đây là floor benchmark:
Warm RMSE : 1.2134  ← con số để beat ở stages sau
Warm MAE  : 0.9251
11.49% predictions sai hơn 2 sao — toàn bộ đến từ ★1 và ★2 bị predict thành ~4. Stages sau (MF, CF) phải cải thiện đáng kể trên low-rating predictions để được coi là thực sự tốt hơn baseline.

In [10]:
import json
from pathlib import Path

ARTIFACTS_DIR = Path("/kaggle/working/stage0_artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

warm = test[mask_warm]

# Tính trước để không lặp lại lỗi
warm_rmse = mean_squared_error(warm["actual"], warm["pred"]) ** 0.5
warm_mae  = mean_absolute_error(warm["actual"], warm["pred"])
all_rmse  = mean_squared_error(test["actual"], test["pred"]) ** 0.5
all_mae   = mean_absolute_error(test["actual"], test["pred"])

# Split meta
split_meta = {
    "category"   : CATEGORY,
    "split_date" : str(split_date),
    "split_idx"  : int(split_idx),
    "n_train"    : len(train),
    "n_test"     : len(test),
    "global_mean": float(global_mean),
    "global_std" : float(global_std),
    "m_user"     : float(m_user),
    "m_item"     : float(m_item),
    "k_core"     : 5,
}
with open(ARTIFACTS_DIR / "split_meta.json", "w") as f:
    json.dump(split_meta, f, indent=2)

# Means
user_stats[["user_mean","user_mean_raw","user_n"]].to_parquet(
    ARTIFACTS_DIR / "user_means.parquet", compression="zstd"
)
item_stats[["item_mean","item_mean_raw","item_n"]].to_parquet(
    ARTIFACTS_DIR / "item_means.parquet", compression="zstd"
)

# Scores
scores = {
    "model"        : "simple_average_baseline",
    "category"     : CATEGORY,
    "warm_rmse"    : float(warm_rmse),
    "warm_mae"     : float(warm_mae),
    "all_rmse"     : float(all_rmse),
    "all_mae"      : float(all_mae),
    "n_warm_test"  : int(mask_warm.sum()),
    "n_total_test" : int(len(test)),
}
with open(ARTIFACTS_DIR / "baseline_scores.json", "w") as f:
    json.dump(scores, f, indent=2)

print("✓ split_meta.json")
print("✓ user_means.parquet")
print("✓ item_means.parquet")
print("✓ baseline_scores.json")
print(f"\nWarm RMSE : {warm_rmse:.4f}")
print(f"Warm MAE  : {warm_mae:.4f}")
print(f"All  RMSE : {all_rmse:.4f}")
print(f"All  MAE  : {all_mae:.4f}")
print(f"\nStage 0 complete — {CATEGORY.upper()}")

✓ split_meta.json
✓ user_means.parquet
✓ item_means.parquet
✓ baseline_scores.json

Warm RMSE : 1.2134
Warm MAE  : 0.9251
All  RMSE : 1.1976
All  MAE  : 0.9167

Stage 0 complete — ELECTRONICS


Nhận xét Stage 0 — Electronics
Những gì đã làm đúng
Pipeline hoàn chỉnh và không có bug nghiêm trọng. Temporal split được thực hiện đúng theo thời gian thay vì random — đây là điểm nhiều team CS246 hay sai. Bayesian shrinkage được áp dụng thay vì dùng raw mean, xử lý đúng vấn đề users/items có ít reviews trong train set. K-core filtering với k=5 đã được thực hiện iterative đến convergence (10 iterations), đảm bảo đúng định nghĩa k-core.
Kết quả đạt được
Dataset Electronics sau 5-core: 5.5M → 5.5M reviews, 609K users, 145K items, sparsity 99.99%.
Baseline scores làm anchor cho toàn bộ project:

Warm RMSE: 1.2134 — con số duy nhất quan trọng để so sánh stages sau
Warm MAE: 0.9251

Phát hiện quan trọng từ per-star breakdown: model hoàn toàn fail trên ★1 (RMSE 3.11) và ★2 (RMSE 2.14) vì Bayesian means cluster quanh 4.2. Đây không phải bug mà là limitation cố hữu của bias-only model — và là lý do MF ở stages sau phải học được latent factors để differentiate low vs high ratings.
Hạn chế cần acknowledge
Warm rate chỉ đạt 82.4% thay vì >90% mong muốn — do 45% users trong Electronics chỉ có 1–2 reviews tổng, một artifact của dataset chứ không phải pipeline. Cold-user RMSE (1.0986) trông tốt hơn warm một cách phản trực giác, giải thích được bằng selection bias của new users.

In [11]:
# ============================================
# SAVE TRAIN / TEST FOR LATER STAGES
# ============================================

# Chỉ lưu các cột cần cho Stage 2/3
train_to_save = train[[COL_USER, COL_ITEM, COL_RATING, COL_DATE]].copy()

test_to_save = test[[
    COL_USER, COL_ITEM, COL_RATING, COL_DATE,
    "user_known", "item_known"
]].copy()

test_to_save["is_warm"] = mask_warm.values
test_to_save["is_cold_user"] = mask_cold_u.values
test_to_save["is_cold_item"] = mask_cold_i.values
test_to_save["is_cold_both"] = mask_cold_ui.values

train_to_save.to_parquet(
    ARTIFACTS_DIR / "train.parquet",
    compression="zstd",
    index=False
)

test_to_save.to_parquet(
    ARTIFACTS_DIR / "test.parquet",
    compression="zstd",
    index=False
)

print("✓ train.parquet")
print("✓ test.parquet")

✓ train.parquet
✓ test.parquet


In [12]:
df_core[[COL_USER, COL_ITEM, COL_RATING, COL_DATE]].to_parquet(
    ARTIFACTS_DIR / "df_core.parquet",
    compression="zstd",
    index=False
)

print("✓ df_core.parquet")

✓ df_core.parquet


In [13]:
import pandas as pd
import json
from pathlib import Path

ARTIFACTS_DIR = Path("/kaggle/working/stage0_artifacts")

# Tính lại từ đầu
user_review_counts = (
    train.groupby(COL_USER)[COL_ITEM]
    .count()
    .reset_index(name="n_reviews")
)

bins   = [0, 1, 4, 19, float("inf")]
labels = ["new (0-1)", "cold (2-4)", "medium (5-19)", "warm (>=20)"]

user_review_counts["group"] = pd.cut(
    user_review_counts["n_reviews"],
    bins=bins,
    labels=labels
).astype(str)

# Distribution
vc_norm  = user_review_counts["group"].value_counts(normalize=True).reindex(labels)
vc_count = user_review_counts["group"].value_counts().reindex(labels)

print("=== Cold-start Profile — Electronics Train Set ===")
print(f"Total users: {len(user_review_counts):,}\n")
for group in labels:
    ratio = vc_norm[group]
    count = vc_count[group]
    bar   = "█" * int(ratio * 40)
    print(f"{group:>15s}  {ratio:.1%}  ({count:>6,})  {bar}")

new_cold = vc_norm["new (0-1)"] + vc_norm["cold (2-4)"]
print(f"\nNew + Cold : {new_cold:.1%} → "
      f"{'Content/Popularity backbone quan trọng' if new_cold > 0.4 else 'MF viable làm chính'}")
print(f"Medium + Warm: {1-new_cold:.1%}")

# Save
profile_dict = {
    "category"          : CATEGORY,
    "total_train_users" : int(len(user_review_counts)),
    "distribution"      : {k: float(vc_norm[k]) for k in labels},
    "counts"            : {k: int(vc_count[k])  for k in labels},
}
with open(ARTIFACTS_DIR / "coldstart_profile.json", "w") as f:
    json.dump(profile_dict, f, indent=2)

user_review_counts.to_parquet(
    ARTIFACTS_DIR / "user_groups.parquet", compression="zstd"
)

print("\n✓ coldstart_profile.json")
print("✓ user_groups.parquet")
print(f"\nStage 1 complete — {CATEGORY.upper()}")

=== Cold-start Profile — Electronics Train Set ===
Total users: 586,014

      new (0-1)  2.9%  (16,877)  █
     cold (2-4)  20.6%  (120,655)  ████████
  medium (5-19)  72.6%  (425,651)  █████████████████████████████
    warm (>=20)  3.9%  (22,831)  █

New + Cold : 23.5% → MF viable làm chính
Medium + Warm: 76.5%

✓ coldstart_profile.json
✓ user_groups.parquet

Stage 1 complete — ELECTRONICS


# Stage 2

In [14]:
# ============================================================
# STAGE 2 — Cell 1: Reload data với text columns
# Chỉ load các cột cần thiết để tiết kiệm RAM
# ============================================================
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import re

NEEDED_S2 = ["asin", "reviewerID", "reviewText", "text", "title", "brand", "price_clean"]

dfs_s2 = []
for path, _ in downloaded:
    match = re.search(r'overall=(\d)', path)
    overall_val = int(match.group(1)) if match else None
    pf = pq.ParquetFile(path)
    df_chunk = pf.read(columns=NEEDED_S2).to_pandas()
    df_chunk[COL_RATING] = overall_val
    dfs_s2.append(df_chunk)

df_full = pd.concat(dfs_s2, ignore_index=True)

# Chỉ giữ lại rows có trong df_core (đã qua 5-core)
core_pairs = set(zip(df_core[COL_USER], df_core[COL_ITEM]))
mask = pd.Series(
    zip(df_full[COL_USER], df_full[COL_ITEM])
).isin(core_pairs)
df_full = df_full[mask.values].reset_index(drop=True)

print(f"df_full shape : {df_full.shape}")
print(f"RAM           : ~{df_full.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"Null reviewText: {df_full['reviewText'].isna().sum():,}")
print(f"Null text      : {df_full['text'].isna().sum():,}")
print(f"Null title     : {df_full['title'].isna().sum():,}")
print(f"Sample text col: {df_full['text'].dropna().iloc[0][:100]}")

df_full shape : (5498333, 8)
RAM           : ~9.62 GB
Null reviewText: 0
Null text      : 0
Null title     : 0
Sample text col: this 16gb wifi enabled acer iconia tab a500 provides the ultimate tablet experience with a gleaming 


In [15]:
import gc

# Giữ lại đúng những gì cần, drop phần còn lại
# df_full cần giữ nhưng drop các cột không dùng trong Stage 2
KEEP_COLS = [COL_USER, COL_ITEM, COL_RATING,
             "reviewText", "text", "title", "brand", "price_clean"]

df_full = df_full[KEEP_COLS]

# df_core không cần nữa — mọi filtering đã xong
del df_core
gc.collect()

# Kiểm tra RAM sau khi dọn
import psutil
ram = psutil.virtual_memory()
print(f"RAM used  : {ram.used/1e9:.1f} GB")
print(f"RAM avail : {ram.available/1e9:.1f} GB")
print(f"RAM total : {ram.total/1e9:.1f} GB")
print(f"\ndf_full shape : {df_full.shape}")
print(f"df_full RAM   : ~{df_full.memory_usage(deep=True).sum()/1e9:.2f} GB")

# text column: confirm là product description
print(f"\n=== text column sample ===")
print(df_full["text"].dropna().iloc[0][:200])
print(f"\n=== title column sample ===")
print(df_full["title"].dropna().iloc[0][:100])
print(f"\n=== brand sample ===")
print(df_full["brand"].value_counts().head(10))

RAM used  : 15.9 GB
RAM avail : 17.3 GB
RAM total : 33.7 GB

df_full shape : (5498333, 8)
df_full RAM   : ~9.62 GB

=== text column sample ===
this 16gb wifi enabled acer iconia tab a500 provides the ultimate tablet experience with a gleaming 10 1 touch screen and attention grabbing metallic design fueled by top of the line technology and th

=== title column sample ===
Acer Iconia Tab A500-10S16u 10.1-Inch Tablet Computer (Aluminum Metallic)

=== brand sample ===
brand
Samsung            115514
Sony               115433
Logitech           111540
AmazonBasics       102303
SanDisk            100593
Asus                62184
Canon               60852
Belkin              53915
Western Digital     49014
Nikon               42455
Name: count, dtype: int64


In [16]:
# ============================================================
# STAGE 2 — Cell 2: Item Profiles — TF-IDF
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
import numpy as np
import gc

# Chỉ dùng train rows
train_ids = set(zip(train[COL_USER], train[COL_ITEM]))
mask_train = pd.Series(
    zip(df_full[COL_USER], df_full[COL_ITEM])
).isin(train_ids)
train_full = df_full[mask_train.values].copy()
print(f"Train rows với text: {len(train_full):,}")

# ── Phần 1: Aggregate reviewText per item ────────────────────
item_reviews = (
    train_full.groupby(COL_ITEM)["reviewText"]
    .apply(lambda x: " ".join(x.dropna().astype(str)))
    .reset_index(name="aggregated_review")
)
print(f"Items có reviews: {len(item_reviews):,}")

tfidf_review = TfidfVectorizer(
    max_features=3000,
    max_df=0.95,
    min_df=5,
    ngram_range=(1, 2),
    sublinear_tf=True
)
review_matrix = tfidf_review.fit_transform(item_reviews["aggregated_review"])
print(f"Review TF-IDF : {review_matrix.shape}, "
      f"RAM ~{review_matrix.data.nbytes/1e9:.2f} GB")

# ── Phần 2: TF-IDF từ title + text per item ──────────────────
item_meta = (
    train_full[["asin", "title", "text"]]
    .drop_duplicates("asin")
    .set_index("asin")
    .reindex(item_reviews["asin"])   # align với item_reviews order
    .reset_index()
)
item_meta["meta_text"] = (
    item_meta["title"].fillna("") + " " +
    item_meta["text"].fillna("")
).str.strip()

tfidf_meta = TfidfVectorizer(
    max_features=1000,
    min_df=3,
    sublinear_tf=True
)
meta_matrix = tfidf_meta.fit_transform(item_meta["meta_text"])
print(f"Meta TF-IDF   : {meta_matrix.shape}, "
      f"RAM ~{meta_matrix.data.nbytes/1e9:.2f} GB")

# ── Phần 3: Numerical + Brand features ───────────────────────
items = (
    train_full.groupby(COL_ITEM)
    .agg(
        avg_rating  = (COL_RATING, "mean"),
        n_reviews   = (COL_RATING, "count"),
        price_clean = ("price_clean", "first"),
        brand       = ("brand", "first"),
    )
    .reindex(item_reviews["asin"])   # align order
    .reset_index()
)

price_vals   = items["price_clean"].fillna(items["price_clean"].median())
alpha_price  = 1.0 / max(price_vals.mean(), 1e-6)
alpha_rating = 1.0 / max(items["avg_rating"].mean(), 1e-6)

numerical = csr_matrix(np.column_stack([
    price_vals.values * alpha_price,
    items["avg_rating"].fillna(3.5).values * alpha_rating,
    np.log1p(items["n_reviews"].values),
]))

# Top 100 brands → one-hot
top_brands = items["brand"].value_counts().head(100).index
items["brand_clean"] = items["brand"].where(
    items["brand"].isin(top_brands), other="other"
)
brand_dummies = csr_matrix(
    pd.get_dummies(items["brand_clean"], prefix="brand")
    .values.astype(float)
)
print(f"Numerical     : {numerical.shape}")
print(f"Brand dummies : {brand_dummies.shape}")

# ── Ghép toàn bộ ─────────────────────────────────────────────
item_profiles = hstack([
    review_matrix,
    meta_matrix,
    numerical,
    brand_dummies,
]).tocsr()

item_enc_s2 = {asin: idx for idx, asin
               in enumerate(item_reviews["asin"])}

print(f"\nItem profiles : {item_profiles.shape}")
print(f"nnz           : {item_profiles.nnz:,}")
print(f"RAM estimate  : ~{item_profiles.data.nbytes/1e9:.3f} GB")

# Dọn RAM
del train_full, item_meta
gc.collect()

ram = psutil.virtual_memory()
print(f"\nRAM available sau Cell 2: {ram.available/1e9:.1f} GB")

Train rows với text: 4,399,402
Items có reviews: 143,281
Review TF-IDF : (143281, 3000), RAM ~0.63 GB
Meta TF-IDF   : (143281, 1000), RAM ~0.06 GB
Numerical     : (143281, 3)
Brand dummies : (143281, 101)

Item profiles : (143281, 4104)
nnz           : 87,129,925
RAM estimate  : ~0.697 GB

RAM available sau Cell 2: 12.2 GB


In [17]:
# ============================================================
# STAGE 2 — Cell 3: Explicit Utility Matrix, double normalize
# ============================================================
import scipy.sparse as sp

user_list = train[COL_USER].unique()
item_list = train[COL_ITEM].unique()

user_enc_s2    = {u: i for i, u in enumerate(user_list)}
item_enc_s2_cf = {a: i for i, a in enumerate(item_list)}

rows = train[COL_USER].map(user_enc_s2).values
cols = train[COL_ITEM].map(item_enc_s2_cf).values
vals = train[COL_RATING].astype(float).values

M = sp.csr_matrix(
    (vals, (rows, cols)),
    shape=(len(user_list), len(item_list))
)
print(f"Utility matrix : {M.shape}, nnz={M.nnz:,}, fill={M.nnz/(M.shape[0]*M.shape[1]):.4%}")

# Bước 1: trừ user mean (vectorized)
M_csr       = M.copy().astype(float)
user_sums   = np.array(M_csr.sum(axis=1)).flatten()
user_counts = np.diff(M_csr.indptr).astype(float)
user_counts[user_counts == 0] = 1
user_means_vec = user_sums / user_counts

row_indices    = np.repeat(np.arange(M_csr.shape[0]), np.diff(M_csr.indptr))
M_csr.data    -= user_means_vec[row_indices]

# Bước 2: trừ item mean (vectorized)
M_csc       = M_csr.tocsc()
item_sums   = np.array(M_csc.sum(axis=0)).flatten()
item_counts = np.diff(M_csc.indptr).astype(float)
item_counts[item_counts == 0] = 1
item_means_vec = item_sums / item_counts

col_indices    = np.repeat(np.arange(M_csc.shape[1]), np.diff(M_csc.indptr))
M_csc.data    -= item_means_vec[col_indices]

M_norm = M_csc.tocsr()

print(f"M_norm shape   : {M_norm.shape}")
print(f"Mean of data   : {M_norm.data.mean():.8f}  (phải gần 0)")
print(f"Data range     : [{M_norm.data.min():.3f}, {M_norm.data.max():.3f}]")

ram = psutil.virtual_memory()
print(f"RAM available  : {ram.available/1e9:.1f} GB")

Utility matrix : (586014, 143281), nnz=4,382,054, fill=0.0052%
M_norm shape   : (586014, 143281)
Mean of data   : 0.00000000  (phải gần 0)
Data range     : [-8.779, 29.726]
RAM available  : 11.9 GB


In [18]:
# ============================================================
# STAGE 2 — Cell 4: Implicit Matrix
# ============================================================

C = sp.csr_matrix(
    (np.ones(len(rows)), (rows, cols)),
    shape=(len(user_list), len(item_list))
)
print(f"Implicit matrix : {C.shape}, nnz={C.nnz:,}")
assert C.nnz == M.nnz, "Mismatch encoding"
print("✓ Consistent với explicit matrix")

Implicit matrix : (586014, 143281), nnz=4,382,054
✓ Consistent với explicit matrix
